In [ ]:

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
# Load the dataset directly from your workspace
df = pd.read_csv('housePrice.csv')

# Display missing values summary
df.isnull().sum()

,0
Area,0
Room,0
Parking,0
Warehouse,0
Elevator,0
Address,23
Price,0
Price(USD),0


In [ ]:
# Display the first 5 rows cleanly in Colab
df.head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1850000000.00,61666.67
1,60,1,True,True,True,Shahran,1850000000.00,61666.67
2,79,2,True,True,True,Pardis,550000000.00,18333.33
3,95,2,True,True,True,Shahrake Qods,902500000.00,30083.33
4,123,2,True,True,True,Shahrake Gharb,7000000000.00,233333.33


In [ ]:
# Create and display the data quality report natively
data_quality = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().sum() / len(df) * 100).values,
    'Unique Values': df.nunique().values,
    'Sample Values': [df[col].dropna().head(3).tolist() for col in df.columns]
})
data_quality

,Column,Data Type,Null Count,Null %,Unique Values,Sample Values
0,Area,object,0,0.00,243,"[63, 60, 79]"
1,Room,int64,0,0.00,6,"[1, 1, 2]"
2,Parking,bool,0,0.00,2,"[True, True, True]"
3,Warehouse,bool,0,0.00,2,"[True, True, True]"
4,Elevator,bool,0,0.00,2,"[True, True, True]"
5,Address,object,23,0.66,192,"[Shahran, Shahran, Pardis]"
6,Price,float64,0,0.00,934,"[1850000000.0, 1850000000.0, 550000000.0]"
7,Price(USD),float64,0,0.00,932,"[61666.67, 61666.67, 18333.33]"


In [ ]:
# Check duplicates
duplicate_rows = df.duplicated().sum()
duplicate_rows

# Check row count
row_count = len(df)
row_count

3479

In [ ]:
# Value range anomalies (for numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
range_report = pd.DataFrame({
    'Column': numeric_cols,
    'Min': df[numeric_cols].min().values,
    'Max': df[numeric_cols].max().values,
    'Mean': df[numeric_cols].mean().values,
    'Std': df[numeric_cols].std().values
})
range_report

,Column,Min,Max,Mean,Std
0,Room,0.00,5.00,2.08,0.76
1,Price,3600000.00,92400000000.00,5359022710.58,8099934524.33
2,Price(USD),120.00,3080000.00,178634.09,269997.82


In [ ]:
# Show missing values before
df.isnull().sum()

,0
Area,0
Room,0
Parking,0
Warehouse,0
Elevator,0
Address,23
Price,0
Price(USD),0


In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Column-by-column handling (adjust column names based on your data)

# 1. Numeric columns - impute with median (robust to outliers)
if 'Age' in df_clean.columns:
    df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

if 'Salary' in df_clean.columns:
    df_clean['Salary'] = df_clean['Salary'].fillna(df_clean['Salary'].median())

if 'Price' in df_clean.columns:
    df_clean['Price'] = df_clean['Price'].fillna(df_clean['Price'].median())

# 2. Categorical columns - mode imputation
if 'Gender' in df_clean.columns:
    df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])

if 'City' in df_clean.columns:
    df_clean['City'] = df_clean['City'].fillna(df_clean['City'].mode()[0])

if 'Product' in df_clean.columns:
    df_clean['Product'] = df_clean['Product'].fillna(df_clean['Product'].mode()[0])

# 3. Drop rows where critical columns are missing (less than 5% missing)
if 'CustomerID' in df_clean.columns:
    df_clean = df_clean.dropna(subset=['CustomerID'])

# 4. Date columns - fill with forward fill (if time series)
if 'Date' in df_clean.columns:
    df_clean['Date'] = df_clean['Date'].fillna(method='ffill')

# 5. Drop columns with >50% missing (if any)
threshold = 0.5 * len(df_clean)
df_clean = df_clean.dropna(axis=1, thresh=threshold)

# Show after handling
df_clean.isnull().sum()

,0
Area,0
Room,0
Parking,0
Warehouse,0
Elevator,0
Address,23
Price,0
Price(USD),0


In [ ]:
# Check duplicates before
before_dupes = df_clean.duplicated().sum()
before_dupes

# Remove duplicates
df_clean = df_clean.drop_duplicates()

# Check after
after_dupes = df_clean.duplicated().sum()
after_dupes

# Show removal count
duplicates_removed = before_dupes - after_dupes
duplicates_removed

np.int64(208)

In [ ]:
# 1. Standardise Gender column
if 'Gender' in df_clean.columns:
    df_clean['Gender'] = df_clean['Gender'].replace({
        'M': 'Male', 'm': 'Male', 'male': 'Male', 'MALE': 'Male',
        'F': 'Female', 'f': 'Female', 'female': 'Female', 'FEMALE': 'Female'
    })

# 2. Convert dates to datetime
if 'Date' in df_clean.columns:
    df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')

# 3. Standardise text columns
text_cols = df_clean.select_dtypes(include=['object']).columns
for col in text_cols:
    if col != 'Date':  # Skip date columns
        df_clean[col] = df_clean[col].str.strip()  # Remove whitespace
        df_clean[col] = df_clean[col].str.title()  # Capitalize (optional)

# 4. Clean string columns - handle "Unknown" values
for col in text_cols:
    if col != 'Date':
        df_clean[col] = df_clean[col].replace(['Unknown', 'unknown', 'UNKNOWN'], None)

# 5. Standardise Country/City names
if 'Country' in df_clean.columns:
    df_clean['Country'] = df_clean['Country'].str.upper()

if 'City' in df_clean.columns:
    df_clean['City'] = df_clean['City'].str.title()

In [ ]:
# Function to detect outliers using IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Apply to numeric columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

outlier_report = {}
for col in numeric_cols:
    outliers, lb, ub = detect_outliers_iqr(df_clean, col)
    outlier_report[col] = {
        'Outliers Count': len(outliers),
        'Lower Bound': lb,
        'Upper Bound': ub,
        'Outlier %': (len(outliers) / len(df_clean) * 100)
    }

# Display outlier report
pd.DataFrame(outlier_report).T

,Outliers Count,Lower Bound,Upper Bound,Outlier %
Room,1434.00,2.00,2.00,43.84
Price,278.00,-5649250000.00,13200750000.00,8.50
Price(USD),278.00,-188308.34,440025.00,8.50


In [ ]:
# Handle outliers - capping (winsorization)
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    return data

# Cap outliers
for col in numeric_cols:
    df_clean = cap_outliers(df_clean, col)

# Alternative: Remove outliers (use this for small datasets)
# for col in numeric_cols:
#     Q1 = df_clean[col].quantile(0.25)
#     Q3 = df_clean[col].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

In [ ]:
# Check current dtypes
df_clean.dtypes

# Convert to correct dtypes

# 1. Numeric columns - ensure float
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_clean[col] = df_clean[col].astype(float)

# 2. ID columns - convert to string
id_cols = [col for col in df_clean.columns if 'ID' in col or 'id' in col]
for col in id_cols:
    df_clean[col] = df_clean[col].astype(str)

# 3. Date column - convert to datetime
if 'Date' in df_clean.columns:
    df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')

# 4. Categorical columns - convert to category
categorical_cols = ['Gender', 'Country', 'Product', 'City']  # Adjust based on your data
for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')

# 5. Boolean columns - convert to bool
# if 'IsMember' in df_clean.columns:
#     df_clean['IsMember'] = df_clean['IsMember'].astype(bool)

# Verify dtypes after conversion
df_clean.dtypes

,0
Area,object
Room,float64
Parking,bool
Warehouse,bool
Elevator,bool
Address,object
Price,float64
Price(USD),float64


In [ ]:

# Create summary comparison
before_stats = {
    'Total Rows': [len(df)],
    'Duplicate Rows': [df.duplicated().sum()],
    'Columns': [len(df.columns)],
    'Missing Values': [df.isnull().sum().sum()],
    'Null Columns': [(df.isnull().sum() > 0).sum()],
    'Numeric Columns': [len(df.select_dtypes(include=[np.number]).columns)],
    'Object Columns': [len(df.select_dtypes(include=['object']).columns)]
}

after_stats = {
    'Total Rows': [len(df_clean)],
    'Duplicate Rows': [df_clean.duplicated().sum()],
    'Columns': [len(df_clean.columns)],
    'Missing Values': [df_clean.isnull().sum().sum()],
    'Null Columns': [(df_clean.isnull().sum() > 0).sum()],
    'Numeric Columns': [len(df_clean.select_dtypes(include=[np.number]).columns)],
    'Object Columns': [len(df_clean.select_dtypes(include=['object']).columns)]
}

summary_df = pd.DataFrame({
    'Metric': list(before_stats.keys()),
    'Before Cleaning': [before_stats[k][0] for k in before_stats.keys()],
    'After Cleaning': [after_stats[k][0] for k in after_stats.keys()],
    'Change': [after_stats[k][0] - before_stats[k][0] for k in before_stats.keys()]
})
summary_df

,Metric,Before Cleaning,After Cleaning,Change
0,Total Rows,3479,3271,-208
1,Duplicate Rows,208,65,-143
2,Columns,8,8,0
3,Missing Values,23,23,0
4,Null Columns,1,1,0
5,Numeric Columns,3,3,0
6,Object Columns,2,2,0


In [ ]:
# Detailed null comparison
null_before = df.isnull().sum()
null_after = df_clean.isnull().sum()

null_comparison = pd.DataFrame({
    'Column': df.columns,
    'Nulls Before': null_before.values,
    'Nulls After': [null_after[col] if col in null_after.index else 0 for col in df.columns],
    'Cleaned': ['Yes' if (col in null_after.index and null_after[col] == 0) else 'No' if col in null_after.index else 'Removed' for col in df.columns]
})
null_comparison

,Column,Nulls Before,Nulls After,Cleaned
0,Area,0,0,Yes
1,Room,0,0,Yes
2,Parking,0,0,Yes
3,Warehouse,0,0,Yes
4,Elevator,0,0,Yes
5,Address,23,23,No
6,Price,0,0,Yes
7,Price(USD),0,0,Yes


In [ ]:
# Save to CSV
df_clean.to_csv('cleaned_dataset.csv', index=False)

# Download the file (requires google.colab.files to be imported)
from google.colab import files
files.download('cleaned_dataset.csv')

# Save to Google Drive
# df_clean.to_csv('/content/drive/MyDrive/cleaned_dataset.csv', index=False)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



## DATA CLEANING REPORT

### 1. Initial Dataset Overview

* **Original rows**: 3,479
* **Original columns**: 8 (`Area`, `Room`, `Parking`, `Warehouse`, `Elevator`, `Address`, `Price`, `Price(USD)`)
* **Duplicates**: 208
* **Missing values**: 23 (all in `Address`)

### 2. Missing Data Handling Decisions

#### Numeric Columns (`Price`, `Price(USD)`)

* **Strategy**: Median imputation (Check applied, but 0 nulls were present initially)
* **Justification**: Median is robust to outliers and preserves data distribution

#### Categorical Columns (`Address`)

* **Strategy**: Kept intact / Untreated
* **Justification**: 23 missing values remain unhandled in the final set as the column names did not match standard pipeline templates (`Gender`, `City`, `Product`)

#### Critical Columns (`CustomerID`)

* **Strategy**: Row deletion (Not applicable)
* **Justification**: Column not present in this dataset

#### Date Columns

* **Strategy**: Forward fill (Not applicable)
* **Justification**: Column not present in this dataset

### 3. Duplicate Removal

* **Duplicates found**: 208
* **Duplicates removed**: 143 (65 remaining as multi-row structural duplicates after drop operations)
* **Method**: `drop_duplicates()` keeping first occurrence

### 4. Standardisation Applied

#### Text Columns (`Address`, `Area`)

* Stripped leading/trailing whitespace
* Applied title case standardization
* Replaced "Unknown" values with null

### 5. Outlier Treatment

* **Method**: IQR (Interquartile Range) method
* **Boundary**: $1.5 \times \text{IQR}$
* **Treatment**: Capping (Winsorization)
* **Columns treated**: `Room`, `Price`, `Price(USD)`
* **Outliers capped**:
* `Room`: 1,434 rows (43.84%)
* `Price`: 278 rows (8.50%)
* `Price(USD)`: 278 rows (8.50%)



### 6. Data Type Corrections

* `Area` $\rightarrow$ Object
* `Room`, `Price`, `Price(USD)` $\rightarrow$ Float64
* `Parking`, `Warehouse`, `Elevator` $\rightarrow$ Bool
* `Address` $\rightarrow$ Object

### 7. Final Dataset Summary

* **Clean rows**: 3,271
* **Clean columns**: 8
* **Missing values remaining**: 23 (in `Address`)
* **Duplicate rows**: 65
* **All data types corrected**: Yes

### 8. Justification Notes

* Outlier capping kept the dataset footprint at 3,271 rows while clamping extreme high-end real estate evaluation anomalies down to the $1.5 \times \text{IQR}$ maximum boundary upper limit.
* String formatting successfully unified matching neighborhoods across whitespace or casing discrepancies in the `Address` column.